# File 1 — Degradation + Acoustic Characterisation (Colab, CPU)

**What this does:** transcodes UrbanSound8K, ESC-50, and Speech Commands through
all 8 AMR-NB modes, then computes acoustic descriptors and per-band energy ratios.
**Output:** `results_file1.zip` — download this when done.
**Runtime:** ~2–3 hours on Colab CPU. No GPU needed.
**Run next:** File 2 (training) then File 3 (stats).

No datasets to pre-add — everything downloads automatically.

## 0. Setup

In [ ]:
import os, sys, subprocess, json, csv, random
from pathlib import Path
import numpy as np
SEED=42; random.seed(SEED); np.random.seed(SEED)
RESULTS=Path("results"); RESULTS.mkdir(exist_ok=True)
FIGS=RESULTS/"figs"; FIGS.mkdir(exist_ok=True)
DEGRADED=Path("degraded"); DEGRADED.mkdir(exist_ok=True)
TMP=Path("tmp"); TMP.mkdir(exist_ok=True)
AMR_MODES=[4.75,5.15,5.90,6.70,7.40,7.95,10.20,12.20]
KEY_MODES=[4.75,12.20]
TARGET_SR=22050; MAX_DUR=4.0; N_MELS=128; N_FFT=2048; HOP=512
BANDS=[(0,500),(500,1000),(1000,2000),(2000,3400),
       (3400,4000),(4000,8000),(8000,11025)]
print("config OK.")

config OK.


## 1. Install AMR-capable ffmpeg

In [ ]:
subprocess.run("apt-get update -qq && apt-get install -y -qq "
               "libavcodec-extra libopencore-amrnb0 libopencore-amrwb0 ffmpeg",
               shell=True)
enc=subprocess.run(["ffmpeg","-encoders"],capture_output=True,text=True).stdout
if "libopencore_amrnb" not in enc:
    print("apt ffmpeg lacks AMR -> static build ...")
    subprocess.run("wget -q https://johnvansickle.com/ffmpeg/releases/"
        "ffmpeg-release-amd64-static.tar.xz && "
        "tar xf ffmpeg-release-amd64-static.tar.xz",shell=True)
    d=[x for x in os.listdir(".") if x.startswith("ffmpeg-") and x.endswith("-static")]
    if d: os.environ["PATH"]=os.path.abspath(d[0])+":"+os.environ["PATH"]
    enc=subprocess.run(["ffmpeg","-encoders"],capture_output=True,text=True).stdout
assert "libopencore_amrnb" in enc,"AMR-NB encoder missing — enable Internet in Runtime menu"
print("AMR-NB encoder OK.")
for p in ["librosa","soundfile","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",p])
print("deps OK.")

AMR-NB encoder OK.
deps OK.


## 2. Download datasets

In [ ]:
import tarfile, zipfile, urllib.request
from tqdm.auto import tqdm as tqdm_auto

def _dl(url, dest, desc="downloading"):
    if Path(dest).exists(): print(f"  {desc}: cached"); return
    print(f"  {desc} ...")
    class Prog:
        def __call__(self,b,bs,ts):
            if ts>0:
                pct=min(b*bs/ts,1)*100
                print(f"\r  {pct:.0f}%",end="",flush=True)
    urllib.request.urlretrieve(url,dest,Prog()); print()

# UrbanSound8K
US8K="/content/UrbanSound8K"
if not Path(f"{US8K}/metadata/UrbanSound8K.csv").exists():
    _dl("https://zenodo.org/records/1203745/files/UrbanSound8K.tar.gz?download=1",
        "/content/u.tgz","UrbanSound8K (~6GB)")
    ok=False
    try:
        with tarfile.open("/content/u.tgz") as t: t.getmembers(); ok=True
    except Exception as e: print("archive incomplete:",e)
    if ok:
        with tarfile.open("/content/u.tgz") as t: t.extractall("/content")
        os.remove("/content/u.tgz"); print("US8K extracted.")
    else:
        print("US8K download failed — re-run this cell (wget -c resumes).")

# ESC-50
ESC="/content/ESC-50-master"
if not Path(f"{ESC}/meta/esc50.csv").exists():
    _dl("https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip",
        "/content/esc.zip","ESC-50")
    with zipfile.ZipFile("/content/esc.zip") as z: z.extractall("/content")
    os.remove("/content/esc.zip"); print("ESC-50 extracted.")

# Speech Commands
SC="/content/speech_commands"
if not Path(SC).exists() or not list(Path(SC).glob("yes/*.wav")):
    _dl("http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz",
        "/content/sc.tgz","Speech Commands (~2.3GB)")
    Path(SC).mkdir(exist_ok=True)
    with tarfile.open("/content/sc.tgz") as t: t.extractall(SC)
    os.remove("/content/sc.tgz"); print("Speech Commands extracted.")

print(f"US8K OK: {Path(US8K+'/metadata/UrbanSound8K.csv').exists()}")
print(f"ESC-50 OK: {Path(ESC+'/meta/esc50.csv').exists()}")
print(f"SC OK: {len(list(Path(SC+'/yes').glob('*.wav')))} yes clips")

  UrbanSound8K (~6GB) ...
  100%


/tmp/ipykernel_2237/1558638970.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open("/content/u.tgz") as t: t.extractall("/content")


US8K extracted.
  ESC-50 ...

ESC-50 extracted.
  Speech Commands (~2.3GB) ...
  100%


/tmp/ipykernel_2237/1558638970.py:43: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  with tarfile.open("/content/sc.tgz") as t: t.extractall(SC)


Speech Commands extracted.
US8K OK: True
ESC-50 OK: True
SC OK: 4044 yes clips


## 3. Index datasets

In [ ]:
def us8k_rows():
    rows=[]
    with open(f"{US8K}/metadata/UrbanSound8K.csv") as fh:
        for r in csv.DictReader(fh):
            p=f"{US8K}/audio/fold{r['fold']}/{r['slice_file_name']}"
            rows.append((p,r["class"],int(r["fold"])))
    return rows
def esc_rows():
    rows=[]
    with open(f"{ESC}/meta/esc50.csv") as fh:
        for r in csv.DictReader(fh):
            rows.append((f"{ESC}/audio/{r['filename']}",r["category"],int(r["fold"])))
    return rows
def sc_rows():
    KWS=["yes","no","up","down","left","right","on","off","stop","go"]
    rows=[]; gi=0
    for kw in KWS:
        d=Path(SC)/kw
        if not d.exists(): continue
        for f in sorted(d.glob("*.wav"))[:200]:
            rows.append((str(f),kw,1+(gi%5))); gi+=1
    return rows
US8K_ROWS=us8k_rows(); ESC_ROWS=esc_rows(); SC_ROWS=sc_rows()
print(f"US8K={len(US8K_ROWS)} ESC50={len(ESC_ROWS)} SC={len(SC_ROWS)}")

US8K=8732 ESC50=2000 SC=2000


## 4. AMR-NB degradation (parallel, skips cached files)

In [ ]:
import soundfile as sf, librosa
from multiprocessing.pool import ThreadPool
import multiprocessing as mp

def amr_nb(inp,out,kbps):
    stem=str(out)+".amr"
    r1=subprocess.run(["ffmpeg","-y","-i",str(inp),"-ac","1","-ar","8000",
        "-b:a",f"{kbps}k","-c:a","libopencore_amrnb","-f","amr",stem],
        capture_output=True,timeout=20)
    if r1.returncode!=0: raise RuntimeError(f"encode failed {Path(inp).name}")
    r2=subprocess.run(["ffmpeg","-y","-i",stem,"-ar",str(TARGET_SR),"-ac","1",str(out)],
        capture_output=True,timeout=20)
    if os.path.exists(stem): os.remove(stem)
    if r2.returncode!=0: raise RuntimeError(f"decode failed {Path(inp).name}")

def _degrade_one(args):
    p,od,k=args; op=od/Path(p).name
    if op.exists(): return True
    try: amr_nb(p,op,k); return True
    except Exception as e: return False

def degrade(rows,name,modes):
    n_workers=max(1,mp.cpu_count()-1)
    for k in modes:
        od=DEGRADED/name/f"amr{k}"; od.mkdir(parents=True,exist_ok=True)
        args=[(p,od,k) for p,_,_ in rows]
        done=sum(1 for p,_,_ in rows if (od/Path(p).name).exists())
        todo=[(p,od,k) for p,_,_ in rows if not (od/Path(p).name).exists()]
        if not todo: print(f"  {name} AMR{k}: all {done} cached"); continue
        with ThreadPool(n_workers) as pool:
            res=list(tqdm_auto(pool.imap(_degrade_one,todo,chunksize=16),
                total=len(todo),desc=f"{name} AMR{k}",leave=False))
        ok=sum(res); print(f"  {name} AMR{k}: {done+ok} files ready")

degrade(US8K_ROWS,"us8k",AMR_MODES)
degrade(ESC_ROWS,"esc50",KEY_MODES)
degrade(SC_ROWS,"sc",KEY_MODES)
print("degradation complete.")

us8k AMR4.75:   0%|          | 0/8732 [00:00<?, ?it/s]

  us8k AMR4.75: 8732 files ready


us8k AMR5.15:   0%|          | 0/8732 [00:00<?, ?it/s]

  us8k AMR5.15: 8732 files ready


us8k AMR5.9:   0%|          | 0/8732 [00:00<?, ?it/s]

...

  esc50 AMR12.2: 2000 files ready


sc AMR4.75:   0%|          | 0/2000 [00:00<?, ?it/s]

  sc AMR4.75: 2000 files ready


sc AMR12.2:   0%|          | 0/2000 [00:00<?, ?it/s]

  sc AMR12.2: 2000 files ready
degradation complete.


## 5. Acoustic characterisation (parallel)

In [ ]:
def descriptors(y,sr=TARGET_SR):
    return dict(centroid=float(librosa.feature.spectral_centroid(y=y,sr=sr).mean()),
                bandwidth=float(librosa.feature.spectral_bandwidth(y=y,sr=sr).mean()),
                rolloff=float(librosa.feature.spectral_rolloff(y=y,sr=sr).mean()),
                flatness=float(librosa.feature.spectral_flatness(y=y).mean()),
                zcr=float(librosa.feature.zero_crossing_rate(y).mean()),
                rms=float(librosa.feature.rms(y=y).mean()))
def band_energy(y,sr=TARGET_SR):
    S=np.abs(librosa.stft(y,n_fft=1024))**2; f=librosa.fft_frequencies(sr=sr,n_fft=1024)
    return {b:float(S[(f>=b[0])&(f<b[1])].sum()) for b in BANDS}
def hf_fraction(y,sr=TARGET_SR,cut=3400):
    S=np.abs(librosa.stft(y,n_fft=1024))**2; f=librosa.fft_frequencies(sr=sr,n_fft=1024)
    return float(S[f>=cut].sum()/(S.sum()+1e-12))
def delta_db(Ed,Ec):
    return {f"{b[0]}-{b[1]}":float(10*np.log10((Ed[b]+1e-12)/(Ec[b]+1e-12))) for b in BANDS}
def find_key(d,target):
    t=float(target)
    for k in d:
        try:
            if abs(float(k)-t)<0.01: return k
        except: pass
    return None

def _ac_one(args):
    p,c,od,first=args; op=od/Path(p).name
    if not op.exists(): return None
    try:
        yc,_=librosa.load(p,sr=TARGET_SR,duration=MAX_DUR)
        yd,_=librosa.load(str(op),sr=TARGET_SR,duration=MAX_DUR)
        dc=descriptors(yc); dq=descriptors(yd)
        return dict(c=c,desc_delta={k:dq[k]-dc[k] for k in dc},
                    band_db=delta_db(band_energy(yd),band_energy(yc)),
                    hc=float(hf_fraction(yc)) if first else None)
    except: return None

def characterise(rows,name,modes):
    n_workers=max(1,mp.cpu_count()-1); prof={}; Hc={}
    for k in modes:
        od=DEGRADED/name/f"amr{k}"
        if not od.exists():
            alt=DEGRADED/name/f"amr{float(k):.2f}".rstrip("0").rstrip(".")
            if alt.exists(): od=alt
        if not od.exists(): continue
        args=[(p,c,od,k==modes[0]) for p,c,_ in rows]
        with ThreadPool(n_workers) as pool:
            res=list(tqdm_auto(pool.imap(_ac_one,args,chunksize=32),
                total=len(args),desc=f"{name} AMR{k}",leave=False))
        dd=[r["desc_delta"] for r in res if r]
        bd=[r["band_db"] for r in res if r]
        for r in res:
            if r and r["hc"] is not None: Hc.setdefault(r["c"],[]).append(r["hc"])
        prof[k]=dict(
            desc_delta={kk:float(np.mean([d[kk] for d in dd])) for kk in dd[0]} if dd else {},
            band_db={b:float(np.mean([d[b] for d in bd])) for b in bd[0]} if bd else {})
        print(f"  {name} AMR{k}: centroid shift "
              f"{prof[k]['desc_delta'].get('centroid',0):.0f} Hz ({len(dd)} clips)")
    return prof,{c:float(np.mean(v)) for c,v in Hc.items()}

ACO={}; HC={}
for name,rows,modes in [("us8k",US8K_ROWS,AMR_MODES),
                         ("esc50",ESC_ROWS,KEY_MODES),
                         ("sc",SC_ROWS,KEY_MODES)]:
    if not rows: continue
    prof,hc=characterise(rows,name,modes); ACO[name]=prof; HC[name]=hc
    print(f"[{name}] done.")
json.dump(ACO,open(RESULTS/"acoustic_profile.json","w"),indent=2)
json.dump(HC,open(RESULTS/"hf_fraction_per_class.json","w"),indent=2)
print("acoustic_profile.json + hf_fraction_per_class.json saved.")

us8k AMR4.75:   0%|          | 0/8732 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1323
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1764
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1103
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1523
  warnings.warn(


  us8k AMR4.75: centroid shift -915 Hz (8732 clips)


us8k AMR5.15:   0%|          | 0/8732 [00:00<?, ?it/s]

  us8k AMR5.15: centroid shift -912 Hz (8732 clips)


...

  esc50 AMR12.2: centroid shift -1060 Hz (2000 clips)
[esc50] done.


sc AMR4.75:   0%|          | 0/2000 [00:00<?, ?it/s]

  sc AMR4.75: centroid shift -450 Hz (2000 clips)


sc AMR12.2:   0%|          | 0/2000 [00:00<?, ?it/s]

  sc AMR12.2: centroid shift -498 Hz (2000 clips)
[sc] done.
acoustic_profile.json + hf_fraction_per_class.json saved.


## 6. Mismatch + figures

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

# env-vs-speech mismatch
def hi_atten(name,target=4.75):
    if name not in ACO or not ACO[name]: return float("nan")
    key=find_key(ACO[name],target)
    if key is None: return float("nan")
    bd=ACO[name][key]["band_db"]
    vals=[bd[b] for b in bd if int(b.split("-")[0])>=3400]
    return float(np.mean(vals)) if vals else float("nan")
mm={n:hi_atten(n) for n in ACO if ACO[n]}
json.dump({"hi_band_attenuation_db_at_4.75k":mm},
          open(RESULTS/"mismatch_env_vs_speech.json","w"),indent=2)
print(">3.4kHz attenuation @4.75k:")
for n,v in mm.items(): print(f"  {n}: {v:+.1f} dB")

# band-energy figure
fig,ax=plt.subplots(figsize=(6.5,3.6))
for name,col in [("us8k","#4477AA"),("esc50","#EE6677"),("sc","#228833")]:
    if name not in ACO or not ACO[name]: continue
    key=find_key(ACO[name],4.75)
    if key is None: continue
    bd=ACO[name][key]["band_db"]; bands=list(bd.keys())
    ax.plot(range(len(bands)),[bd[b] for b in bands],marker="s",
            label=f"{name} @4.75k",color=col)
ax.axhline(0,c="k",lw=.5); ax.set_xticks(range(len(bands)))
ax.set_xticklabels(bands,rotation=45,fontsize=7)
ax.set_ylabel(r"$\Delta E_b$ (dB)")
ax.set_title("Per-band energy change under AMR-NB 4.75k")
ax.legend(); plt.tight_layout()
plt.savefig(FIGS/"fig_band_energy.png",dpi=150); plt.close()
print("saved fig_band_energy.png")

# difference spectrograms
p=US8K_ROWS[0][0]; yc,_=librosa.load(p,sr=TARGET_SR,duration=MAX_DUR)
fig,axes=plt.subplots(2,3,figsize=(11,5))
def mel(y): return librosa.power_to_db(
    librosa.feature.melspectrogram(y=y,sr=TARGET_SR,n_mels=N_MELS),ref=np.max)
Mc=mel(yc)
import librosa.display
librosa.display.specshow(Mc,sr=TARGET_SR,y_axis="mel",x_axis="time",ax=axes[0,0])
axes[0,0].set_title("clean"); axes[1,0].axis("off")
for j,k in enumerate(KEY_MODES):
    od=DEGRADED/"us8k"/f"amr{k}"
    op=od/Path(p).name
    if not op.exists(): continue
    yd,_=librosa.load(str(op),sr=TARGET_SR,duration=MAX_DUR)
    Md=mel(yd); n=min(Mc.shape[1],Md.shape[1])
    librosa.display.specshow(Md,sr=TARGET_SR,y_axis="mel",x_axis="time",ax=axes[0,j+1])
    axes[0,j+1].set_title(f"AMR {k}k")
    librosa.display.specshow(Md[:,:n]-Mc[:,:n],sr=TARGET_SR,y_axis="mel",
        x_axis="time",ax=axes[1,j+1],cmap="coolwarm")
    axes[1,j+1].set_title(f"diff clean\u2212AMR{k}k")
plt.tight_layout(); plt.savefig(FIGS/"fig_difference_spectrograms.png",dpi=150)
plt.close(); print("saved fig_difference_spectrograms.png")

>3.4kHz attenuation @4.75k:
  us8k: -27.6 dB
  esc50: -29.9 dB
  sc: -7.7 dB
saved fig_band_energy.png
saved fig_difference_spectrograms.png


## 7. Bundle and download

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("results_file1","zip","results")
files.download("results_file1.zip")
print("Download results_file1.zip — upload to File 2 session as input.")
print("Files:",sorted(p.name for p in RESULTS.glob("*")))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download results_file1.zip — upload to File 2 session as input.
Files: ['acoustic_profile.json', 'figs', 'hf_fraction_per_class.json', 'mismatch_env_vs_speech.json']
